# Model Comparison

This notebook evaluates different machine learning algorithms for the credit card fraud detection task.

## Objectives

* Train baseline models
* Compare their performance using appropriate metrics for imbalanced classification
* Select a model for further optimization

## Models Evaluated

* Logistic Regression
* Random Forest
* CatBoost
* XgBoost
* AdaBoost
* Extra Trees
* LightGBM
* Multi Layer Perceptron

The best-performing model will be selected for further hyperparameter tuning.

## Notes



In [8]:
from datapipeline.training.load_data import load_raw_data
from datapipeline.config.mlflow_config import setup_mlflow
import yaml
import pandas as pd
import mlflow
from pathlib import Path
from sklearn.metrics import (roc_auc_score, 
                            balanced_accuracy_score,
                            precision_score,
                            recall_score,
                            f1_score,
                            confusion_matrix,
                            precision_recall_curve,
                            average_precision_score,
                            make_scorer)
from sklearn.model_selection import cross_validate
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from lightgbm import LGBMClassifier
import xgboost as xgb
from sklearn.neural_network import MLPClassifier
from sklearn.utils.class_weight import compute_sample_weight

# Configurations

In [9]:
#primary metric that will be used for model comparison
PRIMARY_METRIC = "Average Precision Score"

In [10]:
config_file_path = '../config.yaml'
mlflow_local_folder = '../mlruns'

In [11]:
#There is a config.yaml file at the project root
#loading config file
config_path = '../config.yaml'
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)
    

# MLflow

In [12]:
experiment_name = config['pipeline']['experiment_name']


In [13]:
setup_mlflow(experiment_name, mlflow_local_folder)

file:/home/rodolfo/Insync/rodolfopcruz2@gmail.com/Google Drive/Estudo/Projetos-Novos/Credit_card_fraud_detection/mlruns


In [14]:
mlflow.start_run(run_name='Model Selection')

<ActiveRun: >

In [15]:
artifacts_dir = Path(config['model_selection']['artifacts_path'])

# Load Dataset


In [17]:
dataset_path = Path(config['feature_engineering']['train_path_feature_engineered'])
target_column = config['clean']['target_column']

In [18]:
df_train = pd.read_parquet(dataset_path)

FileNotFoundError: [Errno 2] No such file or directory: 'data/splits/feature_engineered/train_fe.parquet'

In [ ]:
y_train = df_train[target_column]

In [ ]:
x_train = df_train.drop(columns = ['Time', target_column])

# Models

In [ ]:
random_state = config['model_training']['random_state']

In [ ]:
#models that will be tested
dummy = DummyClassifier(strategy = 'most_frequent')
lr = LogisticRegression(max_iter=1000,
                       random_state=random_state, verbose=False)
catboost = CatBoostClassifier(iterations=100, random_state=random_state, verbose=False)
xgboost = xgb.XGBClassifier(n_estimators=100, random_state=random_state)
adaboost = AdaBoostClassifier(n_estimators=100, random_state=random_state)
rf = RandomForestClassifier(n_estimators=100, random_state=random_state, verbose=False)
extra_tree = ExtraTreesClassifier(n_estimators=100 , random_state=random_state, verbose=False)
lgb = LGBMClassifier(n_estimators=100, random_state=random_state)
mlp = MLPClassifier(random_state=random_state, verbose=False)


In [ ]:
candidate_models = {'Dummy': dummy,
                    'Logistic Regression': lr,
                   'CatBoost': catboost,
                   'XgBoost': xgboost,
                   'AdaBoost': adaboost,
                   'Random Forest': rf,
                   'Extra Trees': extra_tree,
                   'LightGBM': lgb, 
                   'MLP': mlp}

In [ ]:
#metrics that will be used for model comparison
metrics = ['Balanced Accuracy Score',
           'Precision Score',
           'Recall Score',
           'F1 Score',
           'Average Precision Score',
           'Roc AUC']


In [ ]:
scoring = {
    'balanced_accuracy': 'balanced_accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'average_precision': 'average_precision',
    'roc_auc': 'roc_auc'
}

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)

# First Tests 

In [ ]:
results_first_tests = pd.DataFrame(columns = metrics,
                       index= candidate_models.keys(),
                       data = 0.0)
results_first_tests

## All selected models will be trained with their default parameter settings, except for the number of estimators used in the ensemble models.

In [ ]:
for model_name, model in candidate_models.items():
    print(f'Training model {model_name}')

    cv_results = cross_validate(model, 
                                x_train, 
                                y_train, 
                                cv=cv, 
                                scoring=scoring, 
                                verbose = False)
   
    metrics_results = [cv_results[f'test_{metric}'].mean() for metric in scoring.keys()]
    print(metrics_results)
    results_first_tests.loc[model_name, :] = metrics_results


In [ ]:
results_first_tests.sort_values(by='Average Precision Score', ascending=False)

In [ ]:
results_first_tests.to_parquet(artifacts_dir / 'first_tests.parquet')

In [ ]:
mlflow.log_artifact(
        artifacts_dir / "first_tests.parquet",
        artifact_path="model_selection"
    )

# Second Tests

## Adjusting a hyperparameter in certain models to address the class imbalance in the dataset.

In [ ]:
candidate_models_second_tests = [
                   'Dummy',
                   'Logistic Regression',
                   'CatBoost',
                   'XgBoost',
                   'AdaBoost',
                   'Random Forest',
                   'Extra Trees',
                   'LightGBM']

In [ ]:
results_second_tests = pd.DataFrame(columns = metrics,
                       index= candidate_models_second_tests,
                       data = 0.0)
results_second_tests

In [ ]:
for model_name in candidate_models_second_tests:

    if model_name == 'Dummy':
        model_adjusted = DummyClassifier(strategy = 'most_frequent')
    
    elif model_name == 'Logistic Regression':
        model_adjusted = LogisticRegression(max_iter=5000,
                                           random_state=random_state,
                                           class_weight='balanced', verbose=False)        
    
    elif model_name == 'CatBoost':
        model_adjusted = CatBoostClassifier(iterations=100,
                                            auto_class_weights='Balanced',
                                            random_state=random_state, verbose=False)

    elif model_name == 'XgBoost':
        n_major = sum(y_train==0)
        n_minor = sum(y_train==1)
        scale_pos_weight = n_major / n_minor

        model_adjusted = xgb.XGBClassifier(n_estimators=100, 
                                           random_state=random_state,
                                           scale_pos_weight=scale_pos_weight)
    elif model_name == 'AdaBoost':
        base_tree = DecisionTreeClassifier(
                    max_depth=1,
                    class_weight='balanced',
                    random_state=random_state)
        model_adjusted = AdaBoostClassifier(n_estimators=100, 
                                      estimator=base_tree,
                                      random_state=random_state)

    elif model_name == 'Random Forest':
        model_adjusted = RandomForestClassifier(n_estimators=100, 
                                    class_weight='balanced',
                                    random_state=random_state,
                                    verbose=False)
                                            
    elif model_name == 'Extra Trees':
        model_adjusted = ExtraTreesClassifier(n_estimators=100,
                                          class_weight='balanced',
                                          random_state=random_state,
                                          verbose=False)                           
    elif model_name == 'LightGBM':
        model_adjusted = LGBMClassifier(n_estimators=100, 
                        random_state=random_state,
                        class_weight='balanced')

   
                                              
    
    print(f'Training model {model_name}')
    cv_results = cross_validate(model_adjusted, x_train, y_train, cv=cv, scoring=scoring, verbose=False)
    metrics_results = [cv_results[f'test_{metric}'].mean() for metric in scoring.keys()]
    print(metrics_results)
    results_second_tests.loc[model_name, :] = metrics_results
    

In [ ]:
results_second_tests.sort_values(by=PRIMARY_METRIC, ascending=False) 

In [ ]:
results_second_tests.to_parquet(artifacts_dir / 'second_tests.parquet')

In [ ]:
mlflow.log_artifact(
        artifacts_dir / "second_tests.parquet",
        artifact_path="model_selection"
    )


# Comparison

## Combining the results of both tests into a single DataFrame

In [ ]:
comparison = pd.concat(
    [results_first_tests, results_second_tests],
    axis=1,
    join='inner',
    keys=['First Tests', 'Second Tests']
)

In [ ]:
comparison.columns.names = ['Tests', 'Metrics']


In [ ]:
comparison = comparison.sort_values(by=[('First Tests','Average Precision Score')], ascending=False)
comparison

In [ ]:
file_name = 'comparison.parquet'
comparison.to_parquet(artifacts_dir / file_name)
mlflow.log_artifact(
        artifacts_dir / file_name,
        artifact_path="model_selection"
    )

## Percentage difference between the second and first tests

In [ ]:
diff_pct = ((comparison['Second Tests'] - comparison['First Tests'])/ comparison['First Tests'])*100

In [ ]:
diff_pct.columns = [f'{column_name} (%)' for column_name in diff_pct.columns]

In [ ]:
diff_pct.style.format('{:.2f}')

In [ ]:
file_name = 'percentage_difference.parquet'
diff_pct.to_parquet(artifacts_dir / file_name)
mlflow.log_artifact(
        artifacts_dir / file_name,
        artifact_path="model_selection"
    )

In [ ]:
mlflow.end_run()